# CIC3: Community Structure Sweep (PSO / Popularity-Similarity)

Investigates how community structure and hubbyness interact in a
topology that has **both** a power-law degree distribution and discrete
macro-communities. The generator is `PSOCommunityGenerator` — a
Papadopoulos-Krioukov popularity-similarity model with a Gaussian
mixture over the angular space that induces `C` discrete communities.

**Motivation.** The README notes observe that `Random` beat `High Degree`
on `Twitter Mutual` but not on any of our synthetic topologies (RSC /
PA / SBM). A plausible explanation is that the effect only appears when
the topology has **both** power-law degrees *and* community structure.
This sweep dials both dimensions independently.

**Sweep axes:**

- **`sigma`** — angular dispersion around each community center. Low
  `sigma` ≈ well-separated communities; high `sigma` ≈ communities
  dissolve and the angular distribution becomes near-uniform, giving a
  pure PSO model with no macro-community structure.
- **`beta`** — popularity-fading parameter. `gamma = 1 + 1/beta`, so
  low `beta` (≈0.2) gives a heavy-tailed power law (strong hubs), and
  `beta → 1` gives a thin tail (weak hubs).

**Fixed:** `N`, `m` (edges per arrival → controls `k_avg`), `m_delta`
(triangle rate → controls `k_delta_avg`), `C` (number of communities,
matched to the number of CIC3 contagions).

**Output:**
1. One heatmap per strategy: `A_g^td` as a function of (beta, sigma).
2. Line sweep: fix `sigma` at a well-separated value and vary `beta`
   (all strategies on one plot); fix `beta` at the default and vary
   `sigma`.
3. Difference heatmaps (strategy − Random) to surface regimes where
   Random dominates.


In [1]:
import sys
sys.path.insert(0, '..')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os

from scm import PSOCommunityGenerator, CIC3Simulator
from scm import (
    MultiRandomSeeding, MultiHighDegreeSeeding, MultiHighSimplexSeeding,
    CommunityLouvainSeeding, CommunityFarthestFirstSeeding,
)
from scm.analysis import (
    attainment, time_discounted_attainment,
    exponential_decay, no_decay,
)

STRATEGIES = [
    ("Random", MultiRandomSeeding),
    ("High Degree", MultiHighDegreeSeeding),
    ("High 2-Simplex", MultiHighSimplexSeeding),
    ("Louvain", CommunityLouvainSeeding),
    ("Farthest-First", CommunityFarthestFirstSeeding),
]

## Shared parameters

In [ ]:
# --- Network ---
N = 500
TOPO_SEED = 2025

# Fixed PSO parameters (k_avg ~ 2*m = 20,  k_delta_avg ~ 3*m_delta = 6)
PSO_M = 10
PSO_M_DELTA = 2

# Number of communities: equal to number of contagions below (C)
# so each contagion can claim a distinct community.
PSO_C_COMMUNITIES = 50

# --- Contagion ---
C = 50
BETA_INFECT = 0.04       # pairwise infection rate (renamed to avoid
BETA_DELTA = 0.03         # clash with PSO beta sweep axis below)
BETAS = [BETA_INFECT] * C
BETA_DELTAS = [BETA_DELTA] * C

# --- Quotas (sum to N, no slack) ---
def make_quotas_and_seeds(N_topo, C=C, seed_frac_divisor=5):
    base = N_topo // C
    rem = N_topo - base * C
    quotas = [base + 1] * rem + [base] * (C - rem)
    seeds = [max(1, q // seed_frac_divisor) for q in quotas]
    assert sum(quotas) == N_topo
    return quotas, seeds

QUOTAS, SEEDS_PER_CONTAGION = make_quotas_and_seeds(N)

# --- Simulation ---
T_MAX = 200
V = no_decay()
NUM_TRIALS = 5

# --- Sweep grid ---
# sigma: community tightness (small = tight communities; large ~pi = dissolved)
SIGMA_VALUES = np.array([0.05, 0.1, 0.2, 0.4, 0.8, 1.5])
# beta: popularity-fading exponent (gamma = 1 + 1/beta)
#   beta=0.2 -> gamma=6 (very heavy tail, strong hubs)
#   beta=0.5 -> gamma=3 (typical social)
#   beta=0.9 -> gamma~2.1 (near-2 power law, very hubby)
BETA_VALUES = np.array([0.2, 0.4, 0.5, 0.7, 0.9])

print(f"Grid: {len(SIGMA_VALUES)} sigma x {len(BETA_VALUES)} beta = "
      f"{len(SIGMA_VALUES) * len(BETA_VALUES)} topology configs")
print(f"Per config: {len(STRATEGIES)} strategies x {NUM_TRIALS} trials")
print(f"Total simulations: "
      f"{len(SIGMA_VALUES) * len(BETA_VALUES) * len(STRATEGIES) * NUM_TRIALS}")

## Topology factory

Build a `PSOCommunityGenerator` for the given (`sigma`, `beta`) grid
point. All other parameters (`N`, `m`, `m_delta`, `C` communities,
community weights) are fixed across the sweep so the only sources of
variation are community tightness and hubbyness.

In [ ]:
def generate_topology(sigma, beta, seed=TOPO_SEED):
    """Generate a PSO-community topology for the given sweep parameters."""
    gen = PSOCommunityGenerator(
        N=N, m=PSO_M, beta=beta,
        C=PSO_C_COMMUNITIES, sigma=sigma,
        m_delta=PSO_M_DELTA,
    )
    links, triangles = gen.generate(seed=seed)
    return {
        "links": links,
        "triangles": triangles,
        "N": N,
        "quotas": QUOTAS,
        "seeds": SEEDS_PER_CONTAGION,
        "k_avg": gen.k_avg,
        "k_d_avg": gen.k_delta_avg,
        "community_labels": gen.community_labels,
    }

## Run the sweep

For each (`sigma`, `beta`) grid point, generate the topology once and run
all strategies for `NUM_TRIALS` each. Results are cached to disk so the
notebook can be re-run without re-computing.

In [ ]:
RESULTS_DIR = "results/pso_sweep"
os.makedirs(RESULTS_DIR, exist_ok=True)


def run_trial(topo, strat_cls):
    """Run a single CIC3 trial and return metrics."""
    seeder = strat_cls(
        N=topo["N"], num_seeds_per_contagion=topo["seeds"],
        links=topo["links"], triangles=topo["triangles"],
    )
    seeds = seeder.seed()
    sim = CIC3Simulator(
        links=topo["links"], triangles=topo["triangles"],
        initial_infected_per_contagion=seeds,
        betas=BETAS, beta_deltas=BETA_DELTAS, quotas=topo["quotas"],
    )
    sim.run(T_MAX)
    A_i, A_g = attainment(sim.infected_by, topo["quotas"])
    A_i_td, A_g_td = time_discounted_attainment(
        sim.infected_by, sim.infection_times, topo["quotas"], V,
    )
    return {"A_g": A_g, "A_g_td": A_g_td, "A_i": A_i, "A_i_td": A_i_td}


def cache_key(sigma, beta, strat_name):
    return f"sigma{sigma:.3f}_beta{beta:.3f}_{strat_name}"


# Master results dict: results[sigma][beta][strat_name] = list of trial dicts
all_results = {}

total_configs = len(SIGMA_VALUES) * len(BETA_VALUES)
config_idx = 0

for sigma in SIGMA_VALUES:
    all_results[sigma] = {}
    for beta in BETA_VALUES:
        config_idx += 1
        print(f"\n[{config_idx}/{total_configs}] "
              f"sigma={sigma:.3f}, beta={beta:.3f}")

        topo = generate_topology(sigma, beta)
        print(f"  Realized k_avg={topo['k_avg']:.2f}, "
              f"k_delta_avg={topo['k_d_avg']:.2f}")

        strat_results = {}
        for strat_name, strat_cls in STRATEGIES:
            key = cache_key(sigma, beta, strat_name)
            cache_path = os.path.join(RESULTS_DIR, f"{key}.pkl")

            if os.path.exists(cache_path):
                with open(cache_path, "rb") as f:
                    trials = pickle.load(f)
                print(f"  [{strat_name}] loaded from cache")
            else:
                trials = []
                for t in range(NUM_TRIALS):
                    trials.append(run_trial(topo, strat_cls))
                with open(cache_path, "wb") as f:
                    pickle.dump(trials, f)
                print(f"  [{strat_name}] {NUM_TRIALS} trials done")

            strat_results[strat_name] = trials
        all_results[sigma][beta] = strat_results

print("\n=== Sweep complete ===")

## Aggregate results

For each (`sigma`, `beta`, strategy) cell, compute mean `A_g^td` across
trials.

In [ ]:
strat_names = [s[0] for s in STRATEGIES]

# Build 2D arrays for heatmaps: shape (len(SIGMA_VALUES), len(BETA_VALUES))
# Convention: rows = sigma (y-axis), cols = beta (x-axis)
heatmaps = {}  # strat_name -> 2D numpy array of mean A_g_td

for strat_name in strat_names:
    grid = np.zeros((len(SIGMA_VALUES), len(BETA_VALUES)))
    for si, sigma in enumerate(SIGMA_VALUES):
        for bi, beta in enumerate(BETA_VALUES):
            trials = all_results[sigma][beta][strat_name]
            mean_td = np.mean([tr["A_g_td"] for tr in trials])
            grid[si, bi] = mean_td
    heatmaps[strat_name] = grid

print("Aggregation done. Heatmap shape:", grid.shape)

## Plot: Heatmaps (one per strategy)

Each heatmap shows mean `A_g^td` with popularity-fading `beta` on the
x-axis (left = strong hubs, right = weak hubs) and community-tightness
`sigma` on the y-axis (bottom = tight communities, top = dissolved).
A shared colorbar makes cross-strategy comparison straightforward.

In [ ]:
vmin = min(h.min() for h in heatmaps.values())
vmax = max(h.max() for h in heatmaps.values())

ncols = 3
nrows = int(np.ceil(len(strat_names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = axes.flatten()

for i, strat_name in enumerate(strat_names):
    ax = axes_flat[i]
    im = ax.imshow(
        heatmaps[strat_name], aspect="auto", origin="lower",
        vmin=vmin, vmax=vmax, cmap="viridis",
        extent=[
            BETA_VALUES[0], BETA_VALUES[-1],
            SIGMA_VALUES[0], SIGMA_VALUES[-1],
        ],
    )
    ax.set_title(strat_name, fontsize=11)
    ax.set_xlabel(r"Popularity-fading $\beta$ (small=hubby)")
    ax.set_ylabel(r"Angular dispersion $\sigma$ (small=tight communities)")

for j in range(len(strat_names), len(axes_flat)):
    axes_flat[j].axis("off")

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label=r"Mean $A_g^{td}$")

fig.suptitle(
    f"CIC3 strategy performance vs PSO community/hubbyness\n"
    f"(N={N}, C={C}, trials={NUM_TRIALS})",
    fontsize=13, y=1.02,
)
fig.tight_layout(rect=[0, 0, 0.88, 1.0])
fig.savefig(
    "../figures/cic3_pso_sweep_heatmaps.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

## Plot: Line sweep — fix sigma (tight communities), vary beta

Fix community tightness at a small `sigma` (well-separated communities)
and sweep hubbyness via `beta`. All strategies plotted together.

In [ ]:
# Pick sigma = 0.1 (tight, well-separated communities) as the fixed point
FIXED_SIGMA = SIGMA_VALUES[1]  # 0.1

strat_colors = plt.cm.tab10(np.linspace(0, 1, 10))[:len(strat_names)]

fig, ax = plt.subplots(figsize=(8, 5))
for si, strat_name in enumerate(strat_names):
    means = []
    stds = []
    for bi, beta in enumerate(BETA_VALUES):
        trials = all_results[FIXED_SIGMA][beta][strat_name]
        vals = [tr["A_g_td"] for tr in trials]
        means.append(np.mean(vals))
        stds.append(np.std(vals, ddof=1) if len(vals) > 1 else 0.0)
    means = np.array(means)
    stds = np.array(stds)
    ax.plot(BETA_VALUES, means, color=strat_colors[si], lw=2,
            marker="o", label=strat_name)
    ax.fill_between(BETA_VALUES, means - stds, means + stds,
                    color=strat_colors[si], alpha=0.15)

ax.set_xlabel(r"Popularity-fading $\beta$ "
              r"($\gamma = 1 + 1/\beta$; small $\beta$ = heavier tail)",
              fontsize=11)
ax.set_ylabel(r"Mean $A_g^{td}$", fontsize=11)
ax.set_title(
    f"Strategy performance vs hubbyness\n"
    rf"(fixed $\sigma={FIXED_SIGMA}$, N={N}, C={C}, trials={NUM_TRIALS})",
    fontsize=12,
)
ax.legend(loc="best")
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(
    "../figures/cic3_pso_sweep_beta_fixed_sigma.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

## Plot: Line sweep — fix beta (typical hubbyness), vary sigma

Fix `beta = 0.5` (gamma ≈ 3, typical social-network exponent) and sweep
community tightness `sigma`. As `sigma` grows, the angular distribution
approaches uniform and the macro-community structure dissolves.

In [ ]:
FIXED_BETA = BETA_VALUES[2]  # 0.5 — gamma ~ 3

fig, ax = plt.subplots(figsize=(8, 5))
for si, strat_name in enumerate(strat_names):
    means = []
    stds = []
    for sgi, sigma in enumerate(SIGMA_VALUES):
        trials = all_results[sigma][FIXED_BETA][strat_name]
        vals = [tr["A_g_td"] for tr in trials]
        means.append(np.mean(vals))
        stds.append(np.std(vals, ddof=1) if len(vals) > 1 else 0.0)
    means = np.array(means)
    stds = np.array(stds)
    ax.plot(SIGMA_VALUES, means, color=strat_colors[si], lw=2,
            marker="o", label=strat_name)
    ax.fill_between(SIGMA_VALUES, means - stds, means + stds,
                    color=strat_colors[si], alpha=0.15)

ax.set_xlabel(r"Angular dispersion $\sigma$ "
              r"(small = tight communities, large = dissolved)",
              fontsize=11)
ax.set_ylabel(r"Mean $A_g^{td}$", fontsize=11)
ax.set_title(
    f"Strategy performance vs community tightness\n"
    rf"(fixed $\beta={FIXED_BETA}$, $\gamma\approx 3$; "
    rf"N={N}, C={C}, trials={NUM_TRIALS})",
    fontsize=12,
)
ax.legend(loc="best")
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(
    "../figures/cic3_pso_sweep_sigma_fixed_beta.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

## Plot: Difference heatmaps (strategy vs Random)

Shows where each non-random strategy beats or loses to the Random baseline.
Positive (blue) = strategy is better; negative (red) = Random is better.

In [ ]:
diff_heatmaps = {}
for strat_name in strat_names:
    if strat_name == "Random":
        continue
    diff_heatmaps[strat_name] = heatmaps[strat_name] - heatmaps["Random"]

diff_max = max(abs(d).max() for d in diff_heatmaps.values())

ncols = 2
nrows = int(np.ceil(len(diff_heatmaps) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes_flat = axes.flatten()

for i, (strat_name, diff) in enumerate(diff_heatmaps.items()):
    ax = axes_flat[i]
    im = ax.imshow(
        diff, aspect="auto", origin="lower",
        vmin=-diff_max, vmax=diff_max, cmap="RdBu",
        extent=[
            BETA_VALUES[0], BETA_VALUES[-1],
            SIGMA_VALUES[0], SIGMA_VALUES[-1],
        ],
    )
    ax.set_title(f"{strat_name} − Random", fontsize=11)
    ax.set_xlabel(r"$\beta$ (small=hubby)")
    ax.set_ylabel(r"$\sigma$ (small=tight communities)")

for j in range(len(diff_heatmaps), len(axes_flat)):
    axes_flat[j].axis("off")

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label=r"$\Delta A_g^{td}$ (strategy − Random)")

fig.suptitle(
    f"Strategy advantage over Random baseline\n"
    f"(blue = strategy better, red = Random better)",
    fontsize=13, y=1.02,
)
fig.tight_layout(rect=[0, 0, 0.88, 1.0])
fig.savefig(
    "../figures/cic3_pso_sweep_diff_heatmaps.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

## Summary table

Print the best strategy for each grid point.

In [ ]:
print(f"{'sigma':<10s}{'beta':<10s}{'Best Strategy':<20s}{'A_g_td':<10s}")
print("-" * 50)
for si, sigma in enumerate(SIGMA_VALUES):
    for bi, beta in enumerate(BETA_VALUES):
        best_strat = None
        best_val = -1
        for strat_name in strat_names:
            val = heatmaps[strat_name][si, bi]
            if val > best_val:
                best_val = val
                best_strat = strat_name
        print(f"{sigma:<10.3f}{beta:<10.3f}{best_strat:<20s}{best_val:<10.4f}")